# Làm sạch dữ liệu Round 3

Notebook này áp dụng các cách xử lý đã chọn trong `BI10_R03_EDA.ipynb` (Phần 9), rồi ghi dữ liệu sạch ra thư mục `clean_data/`. Mỗi bảng là một file CSV. Các notebook phân tích của nhóm chỉ đọc từ thư mục này, không đọc file gốc, để cả nhóm dùng chung một bộ số.

Với mỗi bảng, notebook đi theo thứ tự: nói trước sẽ xử lý gì và vì sao, chạy câu lệnh, rồi in số dòng trước và sau để kiểm tra.

**Cách chạy:**
- Đặt notebook cùng thư mục với `Round3_BI10_dataset`.
- Cài thư viện: `pip install duckdb pandas openpyxl`.
- Chạy từ trên xuống. Nếu đã chạy notebook EDA, notebook này dùng lại file `r3_eda.duckdb` cho nhanh. Nếu chưa có, notebook tự đọc file gốc và tạo file đó.

**Quy ước chung trong các file sạch:**
- Tên cột đều viết thường, ví dụ `skp_salesroom`, `salesroom_province`.
- Ngày giờ ghi dạng `2026-09-15 10:30:00`.
- Cột mới do nhóm thêm vào đều có giải thích ngay ở bảng tương ứng.
- File CSV được ghi theo mã **UTF-8 có BOM** (3 byte đánh dấu ở đầu file), để mở bằng Excel vẫn hiện đúng tiếng Việt.
- **Không mở file CSV sạch bằng Excel rồi lưu lại:** Excel sẽ làm mất số 0 ở đầu các mã như `code_pos` ("040100" thành 40100). Excel cũng không mở hết được file lớn như contract và leads, vì Excel chỉ hiển thị tối đa khoảng 1 triệu dòng. Để đọc bằng Python cho đúng kiểu dữ liệu, dùng hàm `load()` trong file `load_clean.py` (cách dùng ở cuối notebook).

Ô dưới đây chuẩn bị dữ liệu:
- mở file `r3_eda.duckdb`, hoặc tạo file này từ dữ liệu gốc nếu chưa có;
- tạo hàm `q()` để chạy một câu SQL và trả kết quả ra dạng bảng;
- tạo thư mục `clean_data/`;
- tạo hàm `xuat()` để ghi một bảng ra file CSV và in số dòng đã ghi.

In [ ]:
import os, shutil, tempfile, duckdb, pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 120)

DATA = 'Round3_BI10_dataset'
DB = 'r3_eda.duckdb'
OUT = 'clean_data'
TMP = os.path.join(tempfile.gettempdir(), 'duck_r3'); os.makedirs(TMP, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

def build():
    c = duckdb.connect(DB)
    c.execute(f"SET threads=2; SET memory_limit='2GB'; SET temp_directory='{TMP}'; SET preserve_insertion_order=false")
    c.execute(f"CREATE OR REPLACE TABLE contract AS SELECT * FROM read_csv('{DATA}/contract.csv')")
    c.execute(f"CREATE OR REPLACE TABLE leads AS SELECT * FROM read_csv('{DATA}/leads.csv', types={{'code_pos':'VARCHAR','code_sa':'VARCHAR','text_contract_number':'VARCHAR','text_tenor':'VARCHAR','text_accepted_by':'VARCHAR','code_lead_status':'VARCHAR'}})")
    c.execute(f"CREATE OR REPLACE TABLE cp AS SELECT * FROM read_csv('{DATA}/contract_product.csv')")
    c.execute(f"CREATE OR REPLACE TABLE e2s AS SELECT * FROM read_csv('{DATA}/emp_2_salesroom.csv')")
    for f, t in [('salesroom.xlsx', 'salesroom'), ('seller.xlsx', 'seller'), ('cl_lead_fin_reason.xlsx', 'reason')]:
        d = pd.read_excel(f'{DATA}/{f}', dtype=str); c.register('d', d)
        c.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM d"); c.unregister('d')
    c.close()

if not os.path.exists(DB):
    build()
con = duckdb.connect(DB, read_only=True)
con.execute(f"SET threads=2; SET memory_limit='2GB'; SET temp_directory='{TMP}'; SET preserve_insertion_order=false")
def q(sql): return con.execute(sql).df()

def xuat(ten, sql):
    # Ghi CSV mã UTF-8 có BOM (3 byte đánh dấu ở đầu file) để Excel nhận đúng tiếng Việt.
    path = f'{OUT}/{ten}.csv'; tmp = path + '.tmp'
    n = con.execute(f"COPY ({sql}) TO '{tmp}' (HEADER, DELIMITER ',', TIMESTAMPFORMAT '%Y-%m-%d %H:%M:%S', DATEFORMAT '%Y-%m-%d')").fetchone()[0]
    with open(path, 'wb') as f, open(tmp, 'rb') as g:
        f.write(b'\xef\xbb\xbf'); shutil.copyfileobj(g, f, 16 * 1024 * 1024)
    os.remove(tmp)
    print(f'Đã ghi {path}: {n:,} dòng, {os.path.getsize(path)/1e6:,.1f} MB')

SIGNED = "('Active','Finished','Paid off','Written off','Signed')"

## Bảng đổi tên tỉnh cũ sang tỉnh mới

Theo Nghị quyết 202/2025, mỗi tỉnh cũ được gộp vào đúng một tỉnh mới. Có 20 tỉnh cũ còn xuất hiện trong dữ liệu. Ô dưới đây tạo bảng đổi `doi` gồm hai cột: `tinh_cu` và `tinh_moi`. Bảng này dùng cho cả bảng seller và bảng salesroom.

In [ ]:
DOI_TINH = {'Bình Định':'Gia Lai','Bạc Liêu':'Cà Mau','Bắc Giang':'Bắc Ninh','Bắc Kạn':'Thái Nguyên','Bến Tre':'Vĩnh Long',
  'Hà Giang':'Tuyên Quang','Hà Nam':'Ninh Bình','Hải Dương':'Hải Phòng','Kiên Giang':'An Giang','Kon Tum':'Quảng Ngãi',
  'Nam Định':'Ninh Bình','Ninh Thuận':'Khánh Hòa','Phú Yên':'Đắk Lắk','Quảng Bình':'Quảng Trị','Quảng Nam':'Đà Nẵng',
  'Tiền Giang':'Đồng Tháp','Trà Vinh':'Vĩnh Long','Vĩnh Phúc':'Phú Thọ','Yên Bái':'Lào Cai','Đắk Nông':'Lâm Đồng'}
doi = pd.DataFrame(list(DOI_TINH.items()), columns=['tinh_cu','tinh_moi']); con.register('doi', doi)
doi

,tinh_cu,tinh_moi
0,Bình Định,Gia Lai
1,Bạc Liêu,Cà Mau
2,Bắc Giang,Bắc Ninh
3,Bắc Kạn,Thái Nguyên
4,Bến Tre,Vĩnh Long
5,Hà Giang,Tuyên Quang
6,Hà Nam,Ninh Bình
7,Hải Dương,Hải Phòng
8,Kiên Giang,An Giang
9,Kon Tum,Quảng Ngãi


## Bảng 1. seller

**Xử lý:**
- Đổi tên tỉnh cũ sang tỉnh mới. Cột `seller_province` trong file sạch là tên tỉnh theo địa giới mới (34 tỉnh, thành).
- Đổi `seller_active_date` từ dạng chữ "dd-mm-yyyy" sang dạng ngày.

Bảng seller không có dòng trùng hay ô trống, nên không bỏ dòng nào.

Câu lệnh dưới đây tạo bảng seller sạch, rồi so số dòng và số tên tỉnh khác nhau trước và sau xử lý.

In [ ]:
SQL_SELLER = '''
select x.SKP_SELLER::BIGINT skp_seller, x.SELLER_MODEL seller_model, x.SELLER_STATUS seller_status,
  strptime(x.SELLER_ACTIVE_DATE,'%d-%m-%Y %H:%M:%S')::date seller_active_date,
  x.SELLER_TYPE seller_type, x.SELLER_PRODUCT seller_product,
  coalesce(d.tinh_moi, x.SELLER_PROVINCE) seller_province, x.SELLER_AREA seller_area
from seller x left join doi d on x.SELLER_PROVINCE=d.tinh_cu'''
q(f'''select (select count(*) from seller) so_dong_truoc, (select count(*) from ({SQL_SELLER})) so_dong_sau,
  (select count(distinct SELLER_PROVINCE) from seller) so_tinh_truoc, (select count(distinct seller_province) from ({SQL_SELLER})) so_tinh_sau''')

,so_dong_truoc,so_dong_sau,so_tinh_truoc,so_tinh_sau
0,11639,11639,54,34


Số dòng giữ nguyên 11.639. Số tên tỉnh từ 54 về đúng 34 tỉnh, thành. Ghi ra file:

In [ ]:
xuat('seller', SQL_SELLER)

Đã ghi clean_data/seller.csv: 11,639 dòng, 0.8 MB


## Bảng 2. salesroom

**Xử lý:**
- **Đổi tên tỉnh cũ sang tỉnh mới.** Cột `salesroom_province` trong file sạch là tên tỉnh theo địa giới mới (34 tỉnh, thành). Đây là cột dùng làm khu vực khi chia peer group (nhóm shop giống nhau), như đã chọn trong notebook EDA.
- **Thêm cột `salesroom_town_norm`:** tên phường, xã đã bỏ tiền tố "P.", "Phường", "X.", "Xã", "TT.", "Thị trấn" (viết hoa hay thường đều được) và đưa về chữ thường. Riêng phường đánh số như "P. 6", "Phường 06", "PHƯỜNG 6" thì giữ dạng "phường 6", vì nếu chỉ còn con số thì không biết đó là phường. Lưu ý: phường đánh số là phường cũ, cùng một số có thể nằm ở nhiều quận khác nhau, nên khi ghép phải đi kèm cột `salesroom_district`. Dùng cột này khi ghép dữ liệu cấp phường, xã. Cột gốc `salesroom_town` giữ nguyên.
- **Bỏ 15 shop trỏ tới seller không có trong bảng seller.** Cả 15 shop mới xuất hiện ngày 17–18/10/2026, sau khi dữ liệu contract kết thúc, và chưa có hồ sơ, lead hay phân công nhân viên nào (notebook EDA, Kiểm tra 3.3).
- **Đổi `salesroom_start` và `salesroom_end` sang dạng ngày giờ.** Lưu ý: `salesroom_start` là ngày shop xuất hiện trong dữ liệu, không phải ngày shop mở cửa thật.
- Sửa tên cột `SLAESROOM_DISTRICT` trong StructureTable thành `salesroom_district`.
- **Giữ nguyên cột `salesroom_area`** để tham khảo. Vùng kinh doanh đi theo tỉnh cũ, nên một tỉnh mới có thể thuộc nhiều vùng. Đây không phải lỗi. Cột này không dùng để chia peer group.

Câu lệnh dưới đây tạo bảng salesroom sạch, rồi so trước và sau: số dòng, số tên tỉnh, số tên phường xã, và số shop không tìm thấy seller.

In [ ]:
SQL_SALESROOM = '''
select s.SKP_SALESROOM::BIGINT skp_salesroom, s.SALESROOM_CATEGORY salesroom_category, s.SALESROOM_MODEL salesroom_model,
  s.SALESROOM_PRODUCT salesroom_product, s.SALESROOM_STATUS salesroom_status,
  strptime(s.SALESROOM_START,'%d-%m-%Y %H:%M:%S') salesroom_start, strptime(s.SALESROOM_END,'%d-%m-%Y %H:%M:%S') salesroom_end,
  s.SALESROOM_TOWN salesroom_town,
  case when regexp_matches(trim(regexp_replace(trim(s.SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i')),'^[0-9]+$')
       then 'phường ' || cast(cast(trim(regexp_replace(trim(s.SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i')) as int) as varchar)
       else lower(trim(regexp_replace(trim(s.SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i'))) end salesroom_town_norm,
  s.SALESROOM_DISTRICT salesroom_district,
  coalesce(d.tinh_moi, s.SALESROOM_PROVINCE) salesroom_province, s.SALESROOM_AREA salesroom_area,
  s.SKP_EMP_MANAGE::BIGINT skp_emp_manage, s.SKP_SELLER::BIGINT skp_seller
from salesroom s left join doi d on s.SALESROOM_PROVINCE=d.tinh_cu
where s.SKP_SELLER in (select SKP_SELLER from seller)'''
q(f'''select (select count(*) from salesroom) so_dong_truoc, (select count(*) from ({SQL_SALESROOM})) so_dong_sau,
  (select count(distinct SALESROOM_PROVINCE) from salesroom) so_tinh_truoc, (select count(distinct salesroom_province) from ({SQL_SALESROOM})) so_tinh_sau,
  (select count(distinct SALESROOM_TOWN) from salesroom) so_ten_phuong_xa_truoc, (select count(distinct salesroom_town_norm) from ({SQL_SALESROOM})) so_ten_phuong_xa_sau,
  (select count(*) from ({SQL_SALESROOM}) where skp_seller::VARCHAR not in (select SKP_SELLER from seller)) shop_thieu_seller_sau''')

,so_dong_truoc,so_dong_sau,so_tinh_truoc,so_tinh_sau,so_ten_phuong_xa_truoc,so_ten_phuong_xa_sau,shop_thieu_seller_sau
0,14231,14216,54,34,2315,2024,0


Số dòng giảm đúng 15, từ 14.231 xuống 14.216, và không còn shop nào thiếu seller. Số tên tỉnh từ 54 về 34, số tên phường xã từ 2.315 về 2.024. Ghi ra file:

In [ ]:
xuat('salesroom', SQL_SALESROOM)

Đã ghi clean_data/salesroom.csv: 14,216 dòng, 1.8 MB


## Bảng 3. emp_2_salesroom (phân công nhân viên)

**Xử lý:**
- **Bỏ 912 dòng trùng hoàn toàn.** Các dòng này không mang thêm thông tin.
- **Thêm cột `dtime_valid_to_filled`.** Ngày 01/01/3000 ở cột `dtime_valid_to` là giá trị quy ước, nghĩa là phân công chưa kết thúc. Cột mới thay ngày này bằng ngày cuối của dữ liệu (17/10/2026 23:59:59), để tính được thời gian làm việc. Cột gốc giữ nguyên.
- **Thêm cột `is_open`:** bằng 1 nếu phân công còn hiệu lực (`code_status` = a).

Lưu ý khi dùng: nhóm không dùng bảng này để đếm số SA của shop (lý do ở notebook EDA, mục 9.5). Số SA đếm bằng nhân viên khác nhau thực sự làm hồ sơ, từ cột `skp_emp_pre_process` của bảng contract.

Câu lệnh dưới đây tạo bảng sạch, rồi so trước và sau: số dòng, số dòng còn trùng, và thời gian phân công trung bình.

In [ ]:
SQL_E2S = '''
select role, skp_salesroom, skp_employee, dtime_valid_from, dtime_valid_to,
  case when year(dtime_valid_to)=3000 then timestamp '2026-10-17 23:59:59' else dtime_valid_to end dtime_valid_to_filled,
  code_status, (code_status='a')::int is_open
from (select distinct * from e2s)'''
q(f'''select (select count(*) from e2s) so_dong_truoc, (select count(*) from ({SQL_E2S})) so_dong_sau,
  (select count(*) from ({SQL_E2S})) - (select count(*) from (select distinct * from ({SQL_E2S}))) con_trung_sau,
  (select round(avg(date_diff('day',dtime_valid_from,dtime_valid_to))) from e2s) so_ngay_tb_truoc,
  (select round(avg(date_diff('day',dtime_valid_from,dtime_valid_to_filled))) from ({SQL_E2S})) so_ngay_tb_sau''')

,so_dong_truoc,so_dong_sau,con_trung_sau,so_ngay_tb_truoc,so_ngay_tb_sau
0,275856,274944,0,301530.0,255.0


Số dòng giảm từ 275.856 xuống 274.944, không còn dòng trùng. Thời gian phân công trung bình từ hàng trăm nghìn ngày (do ngày 01/01/3000) về vài trăm ngày. Ghi ra file:

In [ ]:
xuat('emp_2_salesroom', SQL_E2S)

Đã ghi clean_data/emp_2_salesroom.csv: 274,944 dòng, 23.6 MB


## Bảng 4. contract

**Xử lý:**
- **Bỏ 7 hồ sơ trống `skp_client`.** Cả 7 đều đang xử lý và thiếu gần hết thông tin.
- **Đổi `date_sign` = 01/01/3000 thành ô trống.** Giá trị này nghĩa là hồ sơ chưa đi tới bước ký. Để trống thì khi đếm theo tháng ký, các hồ sơ này không rơi vào "năm 3000".
- **Thêm các cột dùng chung:**
  - `is_signed`: bằng 1 nếu hồ sơ đã ký, tức `name_status` là Active, Finished, Paid off, Written off hoặc Signed. Đây là quy ước trong StructureTable của Round 2.
  - `is_risk_cohort`: bằng 1 nếu hồ sơ đã ký trong khoảng 03/2025 đến hết 07/2026. Chỉ dùng các hồ sơ này khi tính tỷ lệ trễ hạn, vì hợp đồng ký sau đó chưa đến kỳ trả nên DPD luôn bằng 0, còn hợp đồng ký tháng 01–02/2025 có tỷ lệ trễ thấp bất thường.
  - `apply_month`: tháng nộp hồ sơ, dạng ngày đầu tháng, ví dụ 2026-09-01.
  - `sign_month`: tháng ký, để trống nếu chưa ký.
- **Đổi kiểu dữ liệu:** `skp_emp_pre_process`, `skp_product`, `skp_client`, `payment_num` từ số thập phân sang số nguyên. Các cột DPD sang số nguyên 0/1.

Không thêm cột đánh dấu hợp đồng có nằm ở shop trong bảng salesroom hay không. Khoảng 84% hợp đồng được ghi ở mã điểm bán không có trong bảng salesroom, chủ yếu là vay tiền mặt, thẻ và BNPL ở kênh online. Các hợp đồng này **không bị xóa**, vì cần cho phần cross-sell (bán chéo). Ai cần tách hợp đồng bán trực tiếp tại shop thì tự nối với bảng salesroom theo cột `skp_salesroom`: dùng inner join để chỉ giữ hợp đồng tại shop, left join để giữ tất cả.

Không cộng sẵn tổng tiền cho vay. Với Credit Card và BNPL, `credit_amount` gần như luôn bằng 0, nên số tiền khách dùng nằm ở các cột `amt_billing_1st`, `amt_billing_2nd`, `amt_billing_3rd`. Cách cộng do người làm bước tính KPI quyết định.

Câu lệnh dưới đây tạo bảng sạch, rồi so trước và sau: số dòng, số hồ sơ có ngày ký năm 3000, số hồ sơ đã ký, và số hồ sơ thuộc nhóm dùng để tính rủi ro.

In [ ]:
SQL_CONTRACT = f'''
select c.skp_salesroom, c.skp_credit_case, c.skp_emp_pre_process::BIGINT skp_emp_pre_process, c.skp_product::BIGINT skp_product,
  c.product, c.name_status, c.price, c.init_pay, c.credit_amount, c.ins_amount, c.payment_num::INT payment_num,
  c.skp_client::BIGINT skp_client,
  case when year(c.date_sign)=3000 then null else c.date_sign end date_sign, c.date_apply,
  c.amt_billing_1st, c.amt_billing_2nd, c.amt_billing_3rd,
  c.dpd_1pd::INT dpd_1pd, c.dpd_2pd::INT dpd_2pd, c.dpd_3pd::INT dpd_3pd, c.dpd_4pd::INT dpd_4pd,
  (c.name_status in {SIGNED})::int is_signed,
  (c.name_status in {SIGNED} and c.date_sign>='2025-03-01' and c.date_sign<'2026-08-01')::int is_risk_cohort,
  date_trunc('month', c.date_apply)::date apply_month,
  case when year(c.date_sign)=3000 then null else date_trunc('month', c.date_sign)::date end sign_month
from contract c
where c.skp_client is not null'''
q(f'''select (select count(*) from contract) so_dong_truoc, (select count(*) from ({SQL_CONTRACT})) so_dong_sau,
  (select count(*) from contract where year(date_sign)=3000) ngay_ky_3000_truoc, (select count(*) from ({SQL_CONTRACT}) where year(date_sign)=3000) ngay_ky_3000_sau,
  (select sum(is_signed) from ({SQL_CONTRACT})) da_ky, (select sum(is_risk_cohort) from ({SQL_CONTRACT})) dung_tinh_rui_ro''')

,so_dong_truoc,so_dong_sau,ngay_ky_3000_truoc,ngay_ky_3000_sau,da_ky,dung_tinh_rui_ro
0,3775371,3775364,1048871,0,2715985.0,2180638.0


Số dòng giảm đúng 7, từ 3.775.371 xuống 3.775.364. Không còn hồ sơ nào có ngày ký năm 3000. Có 2,7 triệu hồ sơ đã ký, trong đó khoảng 2,2 triệu thuộc nhóm dùng để tính rủi ro.

Ô dưới đây kiểm tra thêm: tỷ lệ trễ kỳ 1 theo tháng ký, chỉ trên các hồ sơ có `is_risk_cohort` bằng 1. Nếu cột được tạo đúng, các tháng phải nằm trong khoảng ổn định, không còn tháng nào gần 0.

In [ ]:
q(f'''select min(pct) thang_thap_nhat, max(pct) thang_cao_nhat, count(*) so_thang from (
  select sign_month, round(avg(dpd_1pd)*100,2) pct from ({SQL_CONTRACT}) where is_risk_cohort=1 group by 1)''')

,thang_thap_nhat,thang_cao_nhat,so_thang
0,6.76,8.79,17


Tỷ lệ trễ kỳ 1 theo tháng nằm trong khoảng 6,76–8,79% trên 17 tháng, không còn tháng bị lệch. Ghi ra file:

In [ ]:
xuat('contract', SQL_CONTRACT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Đã ghi clean_data/contract.csv: 3,775,364 dòng, 580.3 MB


## Bảng 5. contract_product

**Xử lý:**
- **Giữ một dòng cho mỗi hồ sơ.** Có 3.953 hồ sơ có nhiều dòng. Quy tắc chọn: ưu tiên dòng có `goods_group`, sau đó ưu tiên dòng có loại hàng cụ thể hơn, tức `name_goods_type` không bắt đầu bằng "Other". Lý do: giữ được thông tin chi tiết nhất, và khi nối với contract thì không bị đếm dư hợp đồng.
- **Thêm cột `goods_group_clean`:**
  - gộp các tên chỉ khác nhau ở chữ hoa, chữ thường: "FITNESS" vào "Fitness", "CONSTRUCTION" vào "Construction";
  - gộp "FURNITURE & MATTRESSES" vào "Furniture", "OTHER" vào "Others";
  - ô trống điền theo `name_goods_category`: xe máy mới vào "Two Wheelers", xe máy cũ vào "Used TW", fitness vào "Fitness", còn lại vào "Others".
  - Cột gốc `goods_group` giữ nguyên.

Câu lệnh dưới đây tạo bảng sạch, rồi so trước và sau: số dòng, số hồ sơ, số tên nhóm hàng, số ô trống, và số hợp đồng CD/TW khi nối với contract.

In [ ]:
SQL_CP = '''
select skp_credit_case, name_producer, name_goods_category, name_goods_type, goods_group,
  case
   when goods_group is null and name_goods_category='Two weelers' then 'Two Wheelers'
   when goods_group is null and name_goods_category='USED MOTORBIKE' then 'Used TW'
   when goods_group is null and name_goods_category='FITNESS' then 'Fitness'
   when goods_group is null then 'Others'
   when upper(goods_group) in ('FURNITURE & MATTRESSES','FURNITURE') then 'Furniture'
   when upper(goods_group) in ('OTHER','OTHERS') then 'Others'
   when upper(goods_group)='FITNESS' then 'Fitness'
   when upper(goods_group)='CONSTRUCTION' then 'Construction'
   else goods_group end goods_group_clean
from cp
qualify row_number() over (partition by skp_credit_case order by (goods_group is null), (name_goods_type ilike 'other%'), name_goods_type) = 1'''
q(f'''select (select count(*) from cp) so_dong_truoc, (select count(*) from ({SQL_CP})) so_dong_sau, (select count(distinct skp_credit_case) from ({SQL_CP})) so_ho_so_sau,
  (select count(distinct goods_group) from cp) so_ten_truoc, (select count(distinct goods_group_clean) from ({SQL_CP})) so_ten_sau,
  (select count(*)-count(goods_group_clean) from ({SQL_CP})) o_trong_sau,
  (select count(*) from contract where product in ('CD','TW')) hop_dong_cd_tw_that,
  (select count(*) from contract c join ({SQL_CP}) p using(skp_credit_case) where c.product in ('CD','TW')) sau_khi_noi''')

,so_dong_truoc,so_dong_sau,so_ho_so_sau,so_ten_truoc,so_ten_sau,o_trong_sau,hop_dong_cd_tw_that,sau_khi_noi
0,784902,779941,779941,20,16,0,779943,779941


Số dòng từ 784.902 về 779.941, đúng bằng số hồ sơ: mỗi hồ sơ còn một dòng. Số tên nhóm hàng từ 20 về 16, không còn ô trống. Khi nối với contract, số hợp đồng CD/TW sau nối khớp với số thật (chỉ lệch 2 hồ sơ không có dòng nào trong contract_product). Ghi ra file:

In [ ]:
xuat('contract_product', SQL_CP)

Đã ghi clean_data/contract_product.csv: 779,941 dòng, 55.4 MB


## Bảng 6. leads

**Xử lý:**
- **Không bỏ dòng theo `id_uuid`.** Một mã lead có thể có 2 dòng, thường là cùng một khách được chào cùng lúc hai sản phẩm. Mỗi dòng là một lần chào một sản phẩm.
- **Bỏ 20 lead có `date_assigned` sớm hơn `dtime_created`**, tức được giao trước khi được tạo.
- **Bỏ cột `skp_employee`**, vì trống 100%.
- **Thêm các cột dùng chung:**
  - `result_code`: tên kết quả lead, tra từ cột `result` sang bảng cl_lead_fin_reason. Ví dụ 7 thành DEAL (chốt được), 1 thành NOT_INTEREST (khách không quan tâm).
  - `is_deal`: bằng 1 nếu lead chốt được (`result` = 7).
  - `is_closed`: bằng 1 nếu `code_lead_status` = 4, tức lead đã đóng và có kết quả. Giá trị 1 là lead đang mở.
  - `code_sa_is_person`: bằng 1 nếu `code_sa` là mã số nhân viên. Bằng 0 nếu là XNA hoặc tài khoản hệ thống như "vn_api_fin". Khi tính theo nhân viên chỉ dùng dòng có giá trị 1.
  - `amt_loan_limit_clean`: hạn mức chào, để trống với 10 dòng nhập sai (dưới 1 triệu hoặc trên 300 triệu; ví dụ số hợp đồng bị nhập vào ô hạn mức, xem notebook EDA, Kiểm tra 7.3). Hạn mức từ 250 đến khoảng 281 triệu vẫn giữ, vì hợp lý với khách thu nhập cao. Cột gốc giữ nguyên.
  - `created_month`: tháng tạo lead. Lưu ý: tháng 12/2025 chỉ có 1 ngày dữ liệu và tháng 10/2026 chỉ đến ngày 17.
- **Giữ nguyên `sample_weight`.** Khi đếm số lead phải nhân với cột này.
- **Không thêm cột đánh dấu lead có gắn với shop trong bảng salesroom hay không.** Chỉ khoảng 3% lead (sau khi nhân trọng số) gắn với shop có trong bảng salesroom. Ai cần thì tự nối với bảng salesroom theo cột `skp_salesroom`.

Câu lệnh dưới đây tạo bảng sạch, rồi so trước và sau: số dòng, số lead sau trọng số, số lead DEAL, số dòng `code_sa` là người, và số dòng hạn mức bị để trống.

In [ ]:
SQL_LEADS = '''
select l.id_uuid, l.dtime_created, l.date_assigned, l.id_lead_source, l.text_accepted_by,
  l.amt_loan_limit, case when l.amt_loan_limit<1000000 or l.amt_loan_limit>300000000 then null else l.amt_loan_limit end amt_loan_limit_clean,
  l.code_pos, l.skp_salesroom, l.code_lead_status, (l.code_lead_status='4')::int is_closed,
  l.text_contract_number, l.text_tenor, l.code_sa,
  coalesce(regexp_matches(l.code_sa,'^[0-9]+$'), false)::int code_sa_is_person,
  l.result::INT result, r.CODE_FINISH result_code, (l.result=7)::int is_deal,
  l.skp_credit_case, l.skp_client, l.sample_weight, l.code_product_type_original, l.code_product_type,
  date_trunc('month', l.dtime_created)::date created_month
from leads l
left join reason r on l.result=r.ID_SOURCE::INT
where l.date_assigned >= l.dtime_created'''
q(f'''select (select count(*) from leads) so_dong_truoc, (select count(*) from ({SQL_LEADS})) so_dong_sau,
  (select round(sum(sample_weight)) from leads) lead_trong_so_truoc, (select round(sum(sample_weight)) from ({SQL_LEADS})) lead_trong_so_sau,
  (select sum(is_deal) from ({SQL_LEADS})) lead_deal, (select sum(code_sa_is_person) from ({SQL_LEADS})) code_sa_la_nguoi,
  (select count(*)-count(amt_loan_limit_clean) from ({SQL_LEADS})) han_muc_de_trong''')

,so_dong_truoc,so_dong_sau,lead_trong_so_truoc,lead_trong_so_sau,lead_deal,code_sa_la_nguoi,han_muc_de_trong
0,6130637,6130617,43454845.0,43454657.0,64913.0,3953307.0,19382


Ô dưới đây kiểm tra cột `result_code` đã tra đúng chưa: đếm số dòng theo từng cặp `result` – `result_code`.

In [ ]:
q(f'''select result, result_code, count(*) so_dong from ({SQL_LEADS}) group by all order by 1''')

,result,result_code,so_dong
0,1,NOT_INTEREST,5303725
1,2,EXPIRE_LIFETIME,468235
2,3,REPLACED,247588
3,4,NO_CLUSTER,6175
4,7,DEAL,64913
5,8,REJECTED,15931
6,11,EXISTING_LEAD_ON_LDS,3639
7,<NA>,None,20411


Mỗi mã `result` ứng với đúng một tên kết quả, ví dụ 7 là DEAL, 8 là REJECTED, 1 là NOT_INTEREST. Các dòng trống `result` là lead đang mở nên chưa có kết quả. Ghi ra file (bảng lớn, mất khoảng một phút):

In [ ]:
xuat('leads', SQL_LEADS)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Đã ghi clean_data/leads.csv: 6,130,617 dòng, 1,289.5 MB


## Bảng 7. cl_lead_fin_reason

Bảng tra cứu nhỏ, không có lỗi. Chỉ giữ 3 cột cần dùng: `id_source` (mã), `code_finish` (tên kết quả), `desc_description` (tên dễ đọc). Kết quả tra đã có sẵn trong cột `result_code` của bảng leads sạch, nên bảng này chỉ để tham khảo.

In [ ]:
xuat('cl_lead_fin_reason', '''select ID_SOURCE::INT id_source, CODE_FINISH code_finish, DESC_DESCRIPTION desc_description from reason order by 1''')

Đã ghi clean_data/cl_lead_fin_reason.csv: 12 dòng, 0.0 MB


## Kiểm tra lại các file đã ghi

Ô dưới đây liệt kê các file trong `clean_data/` kèm dung lượng.

In [ ]:
pd.DataFrame([(f, round(os.path.getsize(f'{OUT}/{f}')/1e6,1)) for f in sorted(os.listdir(OUT))], columns=['file','MB'])

,file,MB
0,cl_lead_fin_reason.csv,0.0
1,contract.csv,580.3
2,contract_product.csv,55.4
3,emp_2_salesroom.csv,23.6
4,leads.csv,1289.5
5,salesroom.csv,1.8
6,seller.csv,0.8


## Cách đọc file sạch trong notebook phân tích

File `load_clean.py` (cùng thư mục với notebook này) có hàm `load()`. Hàm này đọc file CSV sạch với đúng kiểu dữ liệu: các mã như `code_pos`, `code_sa`, `id_uuid` giữ dạng chữ để không mất số 0 ở đầu, các cột ngày giờ được đọc thành ngày giờ.

Ô dưới đây thử đọc bảng salesroom bằng hàm này, rồi in kiểu dữ liệu của vài cột để kiểm tra.

In [ ]:
from load_clean import load
sr = load('salesroom')
sr[['skp_salesroom','salesroom_start','salesroom_province','salesroom_area','skp_seller']].dtypes

skp_salesroom                  int64
salesroom_start       datetime64[ns]
salesroom_province            object
salesroom_area                object
skp_seller                     int64
dtype: object

Trong notebook phân tích, chỉ cần đặt notebook cùng thư mục với `load_clean.py` và `clean_data/`, rồi gọi:

```python
from load_clean import load
contract = load('contract')
leads = load('leads')          # bảng lớn, cần khoảng 3–4 GB RAM
```

Nếu máy yếu, không nên đọc cả bảng leads bằng pandas. Thay vào đó dùng DuckDB đọc trực tiếp file CSV và chỉ lấy những cột, những dòng cần dùng:

```python
import duckdb
duckdb.sql("select skp_salesroom, created_month, sum(sample_weight) so_lead from read_csv('clean_data/leads.csv') group by all").df()
```